# Pure Memory Bank Simulation

Simulate the memory bank step by step, **independent of the full model runner**:

1. Pick a stimulus from the real encoded pool → add encoding noise → store in bank
2. Apply drift noise to **all** bank memories (matching the ThreeRegimeNoise schedule)
3. Pick a **novel** stimulus as probe → compute min-distance to bank → FA score
4. Pick a **repeat** stimulus (ISI=1) → compute min-distance → hit score
5. Repeat, growing the bank one item at a time

This isolates the effect of **bank size** on FA/hit scores and d', without
any sequence structure or experimental design confounds.

**Two encoding conditions**: sigma0=13.5 (normal) vs sigma0=0 (noiseless).
The noiseless condition tests whether non-monotonicity is intrinsic to drift+min-of-k.

| Section | What |
|---|---|
| 1 | Setup: encoder, stimuli, noise parameters |
| 2 | Bank simulation function (with pool partitioning to avoid FA collisions) |
| 3 | FA and hit scores vs bank size (sigma0=13.5 vs sigma0=0) |
| 4 | d' vs bank size: local AUROC at each bank size |
| 5 | d' vs sigma1 at fixed bank sizes (KEY PLOT) |
| 6 | Score decomposition: target dist vs best non-target dist |

In [ ]:
import sys, os, yaml, torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

sys.path.append('/om2/user/jmhicks/projects/TextureStreaming/code/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/utls/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/src/model/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/')

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

from utls.runners_v2 import compute_score, ThreeRegimeNoise
from utls.runners_utils import *
from utls.analysis_helpers import auroc_to_dprime
from utls.sigma_fitting import log_mid
from encoders import *

from sklearn.metrics import roc_auc_score

try:
    get_ipython()
    from tqdm.notebook import tqdm, trange
except NameError:
    from tqdm import tqdm, trange

In [ ]:
# --- Load config and encode stimuli ---

def load_config(cfg_path):
    cfg_path = Path(cfg_path)
    if not cfg_path.exists():
        raise FileNotFoundError(cfg_path)
    with open(cfg_path) as f:
        return yaml.safe_load(f), cfg_path

CONFIG_PATH = (
    "/om2/user/bjmedina/auditory-memory/memory/"
    "model_yamls/three-regime/resnet50/nontime_avg/run_000005.yaml"
)
model_cfg, _ = load_config(CONFIG_PATH)

exp_cfg = model_cfg["experiment"]
which_task, is_multi = exp_cfg["which_task"], exp_cfg["is_multi"]
which_isi = exp_cfg.get("which_isi", None)
isis = [0, 1, 2, 4, 8, 16, 32, 64] if is_multi else [0, which_isi]

metric = model_cfg["metric"]
noise_cfg = model_cfg["noise_model"]
noise_mode, t_step = noise_cfg["name"], noise_cfg["t_step"]

repr_cfg = model_cfg["representation"]
time_avg = repr_cfg["time_avg"]
encoder_type = repr_cfg["type"]
layer = repr_cfg.get("layer", None)
pc_dims = repr_cfg.get("pc_dims", None)

exp_list, all_files, name_to_idx, human_runs, task_name, hr_task_name = (
    load_experiment_data(which_task, which_isi, is_multi, old=False)
)
human_curve = compute_human_curve(human_runs, is_multi, which_isi)

# Encode stimuli
NN_ENCODERS = {"kell2018", "resnet50"}
encoder_task = "word_speaker_audioset" if encoder_type in NN_ENCODERS else "audioset"
encoder_cfg = dict(
    encoder_type=encoder_type, model_name=encoder_type, task=encoder_task,
    statistics_dict=statistics_set.statistics, model_params=model_params,
    pc_dims=pc_dims, sr=20000, duration=2.0, rms_level=0.05,
    time_avg=time_avg, device="cuda",
)
if encoder_type in NN_ENCODERS:
    encoder_cfg["layer"] = layer
encoder = build_encoder(encoder_cfg)
X = encode_stimuli(encoder, all_files)

# Model parameters
param_bounds = {"sigma0": (3.0, 22), "sigma1": (0.01, 10.0), "sigma2": (0.0005, 0.5)}
sigma2_init = log_mid(*param_bounds["sigma2"])
SIGMA0 = 13.5  # typical fitted value

# Pre-compute noise scaling factors (same as run_model_core)
dim_std = X.std(dim=0)
rms_std = torch.sqrt(torch.mean(dim_std ** 2))
scaled_std = dim_std / rms_std

print(f"Encoded shape: {X.shape}")
print(f"metric: {metric}, noise_mode: {noise_mode}, t_step: {t_step}")
print(f"sigma0: {SIGMA0}, sigma2_init: {sigma2_init:.6f}")
print(f"dim_std: mean={dim_std.mean():.4f}, rms={rms_std:.4f}")
print(f"Human d' curve: {human_curve}")

---
## 2. Bank Simulation Function

Grow the bank one stimulus at a time. At each step, measure:
- **FA score**: min-distance from a novel probe to all bank memories
- **Hit score**: min-distance from an ISI=1 repeat probe to all bank memories
- **Target distance**: distance from repeat probe to its matching memory only
- **Best non-target distance**: min-distance from repeat probe to all OTHER memories

In [ ]:
def pure_bank_simulation(
    X, sigma0, sigma1, t_step, metric="mahalanobis",
    max_bank_size=30, n_mc=200, seed=42,
):
    """
    Simulate memory bank dynamics step by step.
    
    At each step t (1..max_bank_size):
      - Apply drift noise to all existing memories
      - Pick a novel probe -> compute min-distance -> FA score
      - Pick a repeat probe (re-present item added at step t-1, ISI=1)
        -> compute min-distance -> hit score
      - Add a new stimulus to the bank with encoding noise
    
    Returns dict of lists indexed by bank_size.
    """
    device = X.device
    dtype = X.dtype
    N_stim = X.shape[0]
    
    schedule = ThreeRegimeNoise(sigma0, sigma1, sigma2_init, t_step)
    
    # Collect scores per bank size
    fa_scores_by_k = defaultdict(list)
    hit_scores_by_k = defaultdict(list)
    target_dist_by_k = defaultdict(list)
    nontarget_dist_by_k = defaultdict(list)
    
    for rep in range(n_mc):
        rng = torch.Generator(device=device)
        rng.manual_seed(seed + rep)
        py_rng = np.random.default_rng(seed + rep)
        
        # Shuffle stimulus order for this rep
        stim_order = py_rng.permutation(N_stim)
        
        # PARTITION the pool: first half for bank stimuli, second half for FA probes.
        # This guarantees FA probes are NEVER the same stimulus as anything in the bank.
        n_bank_pool = min(max_bank_size, N_stim // 2)
        bank_stims = stim_order[:n_bank_pool]
        fa_stims = stim_order[n_bank_pool:]
        
        bank = []  # list of {mu, t_inserted, stim_idx}
        
        for t in range(1, n_bank_pool + 2):
            # --- Apply drift noise to ALL existing memories ---
            for mem in bank:
                age = t - mem['t_inserted']
                std = schedule(age)
                noise = torch.randn(
                    mem['mu'].shape, device=device, dtype=dtype, generator=rng
                ) * (std * scaled_std)
                mem['mu'] += noise
            
            # --- Measure scores if bank is non-empty ---
            if len(bank) >= 1:
                k = len(bank)
                
                # FA: novel stimulus from the separate FA pool (never in bank)
                novel_idx = fa_stims[(t - 1) % len(fa_stims)]
                probe_fa = X[novel_idx].view(1, -1)
                
                fa_dists = []
                for mem in bank:
                    age = t - mem['t_inserted']
                    std = schedule(age)
                    score = compute_score(probe_fa, mem['mu'], std, metric)
                    fa_dists.append(score)
                fa_scores_by_k[k].append(min(fa_dists))
                
                # Hit: re-present the most recent stimulus (ISI=1)
                repeat_idx = bank[-1]['stim_idx']
                probe_hit = X[repeat_idx].view(1, -1)
                
                target_d = None
                nontarget_best = float('inf')
                hit_dists = []
                for mem in bank:
                    age = t - mem['t_inserted']
                    std = schedule(age)
                    score = compute_score(probe_hit, mem['mu'], std, metric)
                    hit_dists.append(score)
                    if mem['stim_idx'] == repeat_idx:
                        target_d = score
                    else:
                        nontarget_best = min(nontarget_best, score)
                
                hit_scores_by_k[k].append(min(hit_dists))
                if target_d is not None:
                    target_dist_by_k[k].append(target_d)
                if nontarget_best < float('inf'):
                    nontarget_dist_by_k[k].append(nontarget_best)
            
            # --- Add a new stimulus to the bank ---
            if t <= n_bank_pool:
                stim_idx = bank_stims[t - 1]
                base = X[stim_idx].clone()
                enc_noise = torch.randn(
                    base.shape, device=device, dtype=dtype, generator=rng
                ) * (sigma0 * dim_std)
                mem_new = (base + enc_noise).view(1, -1)
                bank.append({'mu': mem_new, 't_inserted': t, 'stim_idx': int(stim_idx)})
    
    return {
        'fa_scores': dict(fa_scores_by_k),
        'hit_scores': dict(hit_scores_by_k),
        'target_dist': dict(target_dist_by_k),
        'nontarget_dist': dict(nontarget_dist_by_k),
    }


# Run the simulation for several sigma1 values
# Include sigma0=0 (noiseless encoding) as a separate sweep
sigma1_sim_values = [0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 3.0, 10.0]
MAX_BANK = 60
N_MC_SIM = 200

print(f"N_stim = {X.shape[0]}, max_bank_size = {MAX_BANK}")
print(f"Bank pool = {min(MAX_BANK, X.shape[0]//2)} stimuli, "
      f"FA pool = {X.shape[0] - min(MAX_BANK, X.shape[0]//2)} stimuli\n")

# Standard sweep (sigma0=13.5)
sim_results = {}
for s1 in tqdm(sigma1_sim_values, desc="Bank sim (sigma0=13.5)"):
    sim_results[s1] = pure_bank_simulation(
        X, sigma0=SIGMA0, sigma1=s1, t_step=t_step,
        metric=metric, max_bank_size=MAX_BANK, n_mc=N_MC_SIM, seed=2_000_000,
    )

# Noiseless encoding sweep (sigma0=0)
sim_results_noiseless = {}
for s1 in tqdm(sigma1_sim_values, desc="Bank sim (sigma0=0)"):
    sim_results_noiseless[s1] = pure_bank_simulation(
        X, sigma0=0.0, sigma1=s1, t_step=t_step,
        metric=metric, max_bank_size=MAX_BANK, n_mc=N_MC_SIM, seed=2_000_000,
    )

print("Done.")

---
## 3. FA and Hit Scores vs Bank Size

How do the mean and std of FA/hit scores change as the bank grows?

**Two conditions**: sigma0=13.5 (normal encoding noise) vs sigma0=0 (noiseless).
With sigma0=0, performance should start at maximum (perfect encoding). Any
non-monotonicity is purely from the drift + min-of-k interaction.

In [ ]:
# Compare sigma0=13.5 vs sigma0=0 side by side for a few sigma1 values
sigma1_show = [0.01, 0.1, 0.5, 3.0]

fig, axes = plt.subplots(2, len(sigma1_show), figsize=(5 * len(sigma1_show), 10))

for col, s1 in enumerate(sigma1_show):
    for row, (label, results) in enumerate([
        ('sigma0=13.5', sim_results),
        ('sigma0=0', sim_results_noiseless),
    ]):
        res = results[s1]
        bank_sizes = sorted(res['fa_scores'].keys())
        
        fa_means = [np.mean(res['fa_scores'][k]) for k in bank_sizes]
        fa_stds = [np.std(res['fa_scores'][k]) for k in bank_sizes]
        hit_means = [np.mean(res['hit_scores'][k]) for k in bank_sizes]
        hit_stds = [np.std(res['hit_scores'][k]) for k in bank_sizes]
        
        ax = axes[row, col]
        ax.plot(bank_sizes, fa_means, 'o-', ms=3, color='tab:blue', label='FA mean')
        ax.fill_between(bank_sizes,
                        np.array(fa_means) - np.array(fa_stds),
                        np.array(fa_means) + np.array(fa_stds),
                        alpha=0.15, color='tab:blue')
        ax.plot(bank_sizes, hit_means, 'o-', ms=3, color='tab:orange', label='Hit mean')
        ax.fill_between(bank_sizes,
                        np.array(hit_means) - np.array(hit_stds),
                        np.array(hit_means) + np.array(hit_stds),
                        alpha=0.15, color='tab:orange')
        ax.set_xlabel('Bank size')
        ax.set_ylabel('Score (distance)')
        ax.set_title(f'{label}, sigma1={s1}')
        ax.legend(fontsize=7, frameon=False)
        ax.grid(alpha=0.25)

fig.suptitle("Section 3: FA and Hit scores vs bank size\n"
             "Top: normal encoding noise. Bottom: noiseless encoding (sigma0=0).",
             y=1.03, fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# Gap and FA std: sigma0=13.5 (solid) vs sigma0=0 (dashed)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cmap = plt.cm.viridis
colors = cmap(np.linspace(0.05, 0.95, len(sigma1_sim_values)))

for color, s1 in zip(colors, sigma1_sim_values):
    for ls, (label, results) in [('-', ('s0=13.5', sim_results)),
                                  ('--', ('s0=0', sim_results_noiseless))]:
        res = results[s1]
        bank_sizes = sorted(res['fa_scores'].keys())
        
        fa_means = np.array([np.mean(res['fa_scores'][k]) for k in bank_sizes])
        hit_means = np.array([np.mean(res['hit_scores'][k]) for k in bank_sizes])
        gap = fa_means - hit_means
        
        lbl = f's1={s1} {label}' if ls == '-' else None
        axes[0].plot(bank_sizes, gap, ls, ms=2, color=color, label=lbl, alpha=0.8)
        
        fa_stds = np.array([np.std(res['fa_scores'][k]) for k in bank_sizes])
        axes[1].plot(bank_sizes, fa_stds, ls, ms=2, color=color, label=lbl, alpha=0.8)

axes[0].set_xlabel('Bank size')
axes[0].set_ylabel('Gap (FA_mean - Hit_mean)')
axes[0].set_title('Score gap vs bank size\nSolid=sigma0=13.5, Dashed=sigma0=0')
axes[0].legend(fontsize=6, frameon=False, ncol=2)
axes[0].grid(alpha=0.25)

axes[1].set_xlabel('Bank size')
axes[1].set_ylabel('FA std')
axes[1].set_title('FA score std vs bank size\nSolid=sigma0=13.5, Dashed=sigma0=0')
axes[1].legend(fontsize=6, frameon=False, ncol=2)
axes[1].grid(alpha=0.25)

fig.tight_layout()
plt.show()

---
## 4. Local d' vs Bank Size

Compute AUROC (and d') separately at each bank size, using the FA and hit
scores collected at that bank size. This shows how discriminability changes
purely as a function of how many items are in the bank.

In [ ]:
# Local d' vs bank size: sigma0=13.5 (solid) vs sigma0=0 (dashed)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cmap = plt.cm.viridis
colors = cmap(np.linspace(0.05, 0.95, len(sigma1_sim_values)))

for color, s1 in zip(colors, sigma1_sim_values):
    for ls, (label, results) in [('-', ('s0=13.5', sim_results)),
                                  ('--', ('s0=0', sim_results_noiseless))]:
        res = results[s1]
        bank_sizes = sorted(res['fa_scores'].keys())
        
        local_dprime = []
        valid_ks = []
        
        for k in bank_sizes:
            fas = np.array(res['fa_scores'][k])
            hits = np.array(res['hit_scores'][k])
            
            if len(fas) < 5 or len(hits) < 5:
                continue
            
            y_true = np.concatenate([np.ones(len(hits)), np.zeros(len(fas))])
            scores = np.concatenate([hits, fas])
            
            try:
                auroc = roc_auc_score(y_true, -scores)
                dp = auroc_to_dprime(auroc)
            except ValueError:
                continue
            
            valid_ks.append(k)
            local_dprime.append(dp)
        
        lbl = f's1={s1} {label}' if ls == '-' else None
        axes[0].plot(valid_ks, local_dprime, ls, ms=2, color=color, label=lbl, alpha=0.8)

# Also show d' vs bank size for sigma0=0, sigma1=0.01 (should be near-perfect)
axes[0].set_xlabel('Bank size')
axes[0].set_ylabel("d'")
axes[0].set_title("Local d' at each bank size\nSolid=sigma0=13.5, Dashed=sigma0=0 (noiseless)")
axes[0].legend(fontsize=5, frameon=False, ncol=2)
axes[0].grid(alpha=0.25)

# Zoomed view: just sigma0=0 to see noiseless behavior clearly
for color, s1 in zip(colors, sigma1_sim_values):
    res = sim_results_noiseless[s1]
    bank_sizes = sorted(res['fa_scores'].keys())
    
    local_dprime = []
    valid_ks = []
    
    for k in bank_sizes:
        fas = np.array(res['fa_scores'][k])
        hits = np.array(res['hit_scores'][k])
        
        if len(fas) < 5 or len(hits) < 5:
            continue
        
        y_true = np.concatenate([np.ones(len(hits)), np.zeros(len(fas))])
        scores = np.concatenate([hits, fas])
        
        try:
            auroc = roc_auc_score(y_true, -scores)
            dp = auroc_to_dprime(auroc)
        except ValueError:
            continue
        
        valid_ks.append(k)
        local_dprime.append(dp)
    
    axes[1].plot(valid_ks, local_dprime, 'o-', ms=3, color=color, label=f's1={s1}')

axes[1].set_xlabel('Bank size')
axes[1].set_ylabel("d'")
axes[1].set_title("Noiseless encoding (sigma0=0) only\nShows pure drift + min-of-k effect")
axes[1].legend(fontsize=7, frameon=False)
axes[1].grid(alpha=0.25)

fig.suptitle("Section 4: How does discriminability change as bank grows?",
             y=1.03, fontsize=13)
fig.tight_layout()
plt.show()

---
## 5. d' vs sigma1 at Fixed Bank Sizes

This is the key plot: for each bank size, sweep sigma1 and compute d'.
If the non-monotonic bump only appears at small bank sizes and flattens at
large bank sizes, then bank size IS the mechanism.

In [ ]:
# Run a finer sigma1 grid for this analysis — both sigma0 conditions
sigma1_fine = np.exp(np.linspace(np.log(0.01), np.log(10.0), 25))

sim_fine = {}
for s1 in tqdm(sigma1_fine, desc="Fine sweep (sigma0=13.5)"):
    sim_fine[s1] = pure_bank_simulation(
        X, sigma0=SIGMA0, sigma1=s1, t_step=t_step,
        metric=metric, max_bank_size=MAX_BANK, n_mc=N_MC_SIM, seed=3_000_000,
    )

sim_fine_noiseless = {}
for s1 in tqdm(sigma1_fine, desc="Fine sweep (sigma0=0)"):
    sim_fine_noiseless[s1] = pure_bank_simulation(
        X, sigma0=0.0, sigma1=s1, t_step=t_step,
        metric=metric, max_bank_size=MAX_BANK, n_mc=N_MC_SIM, seed=3_000_000,
    )

print("Done.")

In [ ]:
# d' vs sigma1, one curve per bank size — sigma0=13.5 vs sigma0=0

bank_sizes_to_plot = [2, 3, 5, 8, 12, 20, 40, 60]
# Filter to available sizes
bank_sizes_to_plot = [k for k in bank_sizes_to_plot if k <= MAX_BANK]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cmap = plt.cm.coolwarm
colors = cmap(np.linspace(0.05, 0.95, len(bank_sizes_to_plot)))

for ax, (title, sim_data) in zip(axes, [
    ('sigma0=13.5 (normal)', sim_fine),
    ('sigma0=0 (noiseless)', sim_fine_noiseless),
]):
    for color, k in zip(colors, bank_sizes_to_plot):
        dprimes = []
        valid_s1 = []
        
        for s1 in sigma1_fine:
            res = sim_data[s1]
            if k not in res['fa_scores'] or k not in res['hit_scores']:
                continue
            
            fas = np.array(res['fa_scores'][k])
            hits = np.array(res['hit_scores'][k])
            
            if len(fas) < 5 or len(hits) < 5:
                continue
            
            y_true = np.concatenate([np.ones(len(hits)), np.zeros(len(fas))])
            scores = np.concatenate([hits, fas])
            
            try:
                auroc = roc_auc_score(y_true, -scores)
                dp = auroc_to_dprime(auroc)
            except ValueError:
                continue
            
            valid_s1.append(s1)
            dprimes.append(dp)
        
        ax.plot(valid_s1, dprimes, 'o-', ms=4, color=color, label=f'k={k}')
    
    ax.set_xscale('log')
    ax.set_xlabel('sigma1')
    ax.set_ylabel("d'")
    ax.set_title(title)
    ax.legend(fontsize=8, frameon=False)
    ax.grid(alpha=0.25)

fig.suptitle("Section 5: d' vs sigma1 at fixed bank sizes\n"
             "Left: with encoding noise. Right: noiseless encoding (pure drift effect).\n"
             "Blue=small bank, Red=large bank.",
             y=1.07, fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# "Global" d' pooling across bank sizes: sigma0=13.5 vs sigma0=0

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

pool_configs = {
    'all bank sizes': lambda k: True,
    'small bank (k<=5)': lambda k: k <= 5,
    'large bank (k>=10)': lambda k: k >= 10,
}

for ax, (title, sim_data) in zip(axes, [
    ('sigma0=13.5 (normal)', sim_fine),
    ('sigma0=0 (noiseless)', sim_fine_noiseless),
]):
    for label, keep_fn in pool_configs.items():
        dprimes = []
        valid_s1 = []
        
        for s1 in sigma1_fine:
            res = sim_data[s1]
            all_fas = []
            all_hits = []
            
            for k in sorted(res['fa_scores'].keys()):
                if not keep_fn(k):
                    continue
                all_fas.extend(res['fa_scores'][k])
                all_hits.extend(res['hit_scores'].get(k, []))
            
            if len(all_fas) < 10 or len(all_hits) < 10:
                continue
            
            fas = np.array(all_fas)
            hits = np.array(all_hits)
            y_true = np.concatenate([np.ones(len(hits)), np.zeros(len(fas))])
            scores = np.concatenate([hits, fas])
            
            try:
                auroc = roc_auc_score(y_true, -scores)
                dp = auroc_to_dprime(auroc)
            except ValueError:
                continue
            
            valid_s1.append(s1)
            dprimes.append(dp)
        
        ax.plot(valid_s1, dprimes, 'o-', ms=4, label=label)
    
    ax.set_xscale('log')
    ax.set_xlabel('sigma1')
    ax.set_ylabel("d'")
    ax.set_title(title)
    ax.legend(fontsize=9, frameon=False)
    ax.grid(alpha=0.25)

fig.suptitle("d' vs sigma1 pooled across bank sizes\n"
             "If sigma0=0 still shows a bump → it's from drift+min-of-k, not encoding noise",
             y=1.05, fontsize=13)
fig.tight_layout()
plt.show()

---
## 6. Score Decomposition: Target vs Best Non-Target

For ISI=1 hits, the min-distance is either to the **target memory** or to
the **nearest non-target**. As the bank grows, the probability of a non-target
being closer increases (more items → smaller min-of-k). This makes hits
indistinguishable from FAs.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for col, s1 in enumerate(sigma1_sim_values[:4]):
    res = sim_results[s1]
    bank_sizes = sorted(k for k in res['target_dist'].keys() if k >= 2)
    
    target_means = [np.mean(res['target_dist'][k]) for k in bank_sizes]
    nontarget_means = [np.mean(res['nontarget_dist'][k]) for k in bank_sizes]
    
    # Fraction where target wins (is closer than best non-target)
    frac_target_wins = []
    for k in bank_sizes:
        targets = np.array(res['target_dist'][k])
        nontargets = np.array(res['nontarget_dist'][k])
        n = min(len(targets), len(nontargets))
        wins = np.sum(targets[:n] <= nontargets[:n]) / n
        frac_target_wins.append(wins)
    
    ax = axes[0, col]
    ax.plot(bank_sizes, target_means, 'o-', ms=3, color='tab:orange', label='Target dist')
    ax.plot(bank_sizes, nontarget_means, 'o-', ms=3, color='tab:blue', label='Best non-target')
    ax.set_xlabel('Bank size')
    ax.set_ylabel('Mean distance')
    ax.set_title(f'sigma1={s1}')
    ax.legend(fontsize=7, frameon=False)
    ax.grid(alpha=0.25)
    
    ax2 = axes[1, col]
    ax2.plot(bank_sizes, frac_target_wins, 'o-', ms=3, color='tab:green')
    ax2.set_xlabel('Bank size')
    ax2.set_ylabel('Fraction target wins')
    ax2.set_ylim(-0.05, 1.05)
    ax2.axhline(0.5, ls=':', color='gray', alpha=0.5)
    ax2.set_title(f'sigma1={s1}')
    ax2.grid(alpha=0.25)

for col, s1 in enumerate(sigma1_sim_values[4:]):
    res = sim_results[s1]
    bank_sizes = sorted(k for k in res['target_dist'].keys() if k >= 2)
    
    target_means = [np.mean(res['target_dist'][k]) for k in bank_sizes]
    nontarget_means = [np.mean(res['nontarget_dist'][k]) for k in bank_sizes]
    
    frac_target_wins = []
    for k in bank_sizes:
        targets = np.array(res['target_dist'][k])
        nontargets = np.array(res['nontarget_dist'][k])
        n = min(len(targets), len(nontargets))
        wins = np.sum(targets[:n] <= nontargets[:n]) / n
        frac_target_wins.append(wins)
    
    # Reuse row 0 slots for the remaining sigma1 values
    # We need to handle the layout differently
    pass  # handled by the 2x4 grid above

fig.suptitle("Section 6: Target distance vs best non-target vs bank size\n"
             "When non-target drops below target, hits become FAs",
             y=1.03, fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# Cleaner version: target wins fraction vs bank size, all sigma1 on one plot

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cmap = plt.cm.viridis
colors = cmap(np.linspace(0.05, 0.95, len(sigma1_sim_values)))

for color, s1 in zip(colors, sigma1_sim_values):
    res = sim_results[s1]
    bank_sizes = sorted(k for k in res['target_dist'].keys() if k >= 2)
    
    frac_target_wins = []
    for k in bank_sizes:
        targets = np.array(res['target_dist'][k])
        nontargets = np.array(res['nontarget_dist'][k])
        n = min(len(targets), len(nontargets))
        if n == 0:
            continue
        wins = np.sum(targets[:n] <= nontargets[:n]) / n
        frac_target_wins.append(wins)
    
    valid_ks = bank_sizes[:len(frac_target_wins)]
    axes[0].plot(valid_ks, frac_target_wins, 'o-', ms=3, color=color, label=f's1={s1}')
    
    # Gap between target and non-target means
    target_means = [np.mean(res['target_dist'][k]) for k in bank_sizes]
    nontarget_means = [np.mean(res['nontarget_dist'][k]) for k in bank_sizes]
    gap = np.array(nontarget_means) - np.array(target_means)
    axes[1].plot(bank_sizes, gap, 'o-', ms=3, color=color, label=f's1={s1}')

axes[0].set_xlabel('Bank size')
axes[0].set_ylabel('Fraction target wins')
axes[0].set_title('How often does the TARGET memory produce the min score?\n'
                   'Drops as bank grows (more non-target competitors)')
axes[0].axhline(0.5, ls=':', color='gray', alpha=0.5)
axes[0].legend(fontsize=7, frameon=False)
axes[0].grid(alpha=0.25)

axes[1].set_xlabel('Bank size')
axes[1].set_ylabel('Non-target mean - Target mean')
axes[1].set_title('Gap: non-target minus target distance\n'
                   'Positive = target closer (good). Negative = non-target wins.')
axes[1].axhline(0, ls=':', color='gray', alpha=0.5)
axes[1].legend(fontsize=7, frameon=False)
axes[1].grid(alpha=0.25)

fig.tight_layout()
plt.show()

---
## Summary

**Fill in after running:**

1. **FA scores vs bank size** (Section 3): FA mean [decreases / stays flat] as
   bank grows. With sigma0=0, hit scores at low sigma1 should be near-zero
   (perfect match). FA std [increases / U-shape / decreases].

2. **Local d' vs bank size** (Section 4): d' [decreases monotonically / non-monotonic]
   as bank grows. sigma0=0 should show [higher / same] d' than sigma0=13.5 at all
   bank sizes. Does the noiseless condition reach a ceiling?

3. **d' vs sigma1 at fixed bank sizes** (Section 5):
   - **sigma0=13.5**: At k=2 (small bank): d' vs sigma1 is [monotonic / non-monotonic]
   - **sigma0=0**: At k=2: d' [starts at maximum and decays / still has a bump]
   - If sigma0=0 starts at max and monotonically decays, the bump in sigma0=13.5
     is caused by encoding noise creating a floor that drift initially helps overcome.
   - If sigma0=0 ALSO has a bump, the mechanism is intrinsic to drift+min-of-k.

4. **Target vs non-target** (Section 6): At small banks, target wins [___]% of
   the time. At large banks, target wins [___]%. The crossover happens at k=[___].

**Conclusion**: The d'(ISI=1) non-monotonicity [is / is not] explained by [___].